In [ ]:
!pip install tabpfn

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import random
import numpy as np
import torch
import pandas as pd
import os
import sys
import shutil
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from tabpfn import TabPFNClassifier

In [ ]:
def set_seed(seed=42):
    """
    Configura la semilla en todas las librerías para garantizar
    la reproducibilidad absoluta del laboratorio.
    """
    # 1. Python nativo
    random.seed(seed)

    # 2. NumPy (operaciones matriciales y EDA)
    np.random.seed(seed)

    # 3. PyTorch (Backend que utiliza TabPFN)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # Por si ejecutan en entornos multi-GPU

    # 4. Forzar a PyTorch a usar algoritmos deterministas
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f"Semilla global configurada en {seed}.")

# Ejecutar la función antes de preparar los datos
set_seed(42)

In [ ]:

def descargar_y_guardar_dataset(kaggle_id, nombre_archivo_local):
    """
    Descarga un dataset de Kaggle y guarda copia local.
    Si falla la descarga, usa el respaldo local.
    """
    os.makedirs("datasets_respaldo", exist_ok=True)
    ruta_respaldo = os.path.join("datasets_respaldo", nombre_archivo_local)

    try:
        print(f"Descargando '{kaggle_id}' desde Kaggle...")
        path = kagglehub.dataset_download(kaggle_id)

        archivo_csv = [f for f in os.listdir(path) if f.endswith('.csv')][0]
        ruta_original = os.path.join(path, archivo_csv)
        shutil.copy(ruta_original, ruta_respaldo)
        print(f"Dataset guardado en: {ruta_respaldo}")
        return pd.read_csv(ruta_respaldo)

    except Exception as e:
        print(f"Error al descargar: {e}")
        print("Buscando respaldo local...")

        if os.path.exists(ruta_respaldo):
            print(f"Respaldo encontrado: {ruta_respaldo}")
            return pd.read_csv(ruta_respaldo)
        else:
            raise FileNotFoundError(
                "No se pudo descargar el dataset y no hay respaldo local. "
                "Verifica la conexión a internet para la primera descarga."
            )

# ---------------------------------------------------------
# Crop Recommendation Dataset
# ---------------------------------------------------------
print("--- DATASET 1: CROP RECOMMENDATION ---")
df_crop = descargar_y_guardar_dataset(
    kaggle_id="atharvaingle/crop-recommendation-dataset",
    nombre_archivo_local="crop_recommendation.csv"
)

if df_crop is not None:
    print("\nVista previa:")
    display(df_crop.head(2))

# ---------------------------------------------------------
# German Credit Dataset
# ---------------------------------------------------------
def cargar_german_credit():
    """
    Carga German Credit con Risk desde Kaggle.
    Si falla, usa respaldo local.
    """
    os.makedirs("datasets_respaldo", exist_ok=True)
    ruta_respaldo = os.path.join("datasets_respaldo", "german_credit_with_risk.csv")

    try:
        print("Descargando 'kabure/german-credit-data-with-risk' desde Kaggle...")
        df = kagglehub.load_dataset(
            KaggleDatasetAdapter.PANDAS,
            "kabure/german-credit-data-with-risk",
            ""
        )
        df.to_csv(ruta_respaldo, index=False)
        print("Descarga exitosa.")
        return df
    except Exception as e:
        print(f"Error al descargar: {e}")
        print("Buscando respaldo local...")
        if os.path.exists(ruta_respaldo):
            print(f"Respaldo encontrado: {ruta_respaldo}")
            return pd.read_csv(ruta_respaldo)
        raise FileNotFoundError("No hay internet ni respaldo local.")

print("\n--- DATASET 2: GERMAN CREDIT ---")
df_german = cargar_german_credit()

if df_german is not None:
    print("\nVista previa:")
    display(df_german.head(2))


In [ ]:
print("Preprocesando German Credit...")

# 1. Limpiar columna de índice si existe
if 'Unnamed: 0' in df_german.columns:
    df_german = df_german.drop(columns=['Unnamed: 0'])

# 2. Columna objetivo
target_col = 'Risk'

print(f"Columna objetivo: '{target_col}'")

df_processed = df_german.copy()
df_processed = df_processed.dropna(subset=[target_col])

X = df_processed.drop(columns=[target_col])
y = df_processed[target_col]

# 3. Codificar variable objetivo
le = LabelEncoder()
y = le.fit_transform(y)
print(f"Clases detectadas: {le.classes_} -> [0, 1]")

# 4. Manejo de valores nulos
categorical_cols = X.select_dtypes(include=['object', 'category']).columns
numeric_cols = X.select_dtypes(include=['number']).columns

if len(categorical_cols) > 0:
    X[categorical_cols] = X[categorical_cols].fillna('Desconocido')
if len(numeric_cols) > 0:
    X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

# 5. Codificar categóricas para TabPFN
if len(categorical_cols) > 0:
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X[categorical_cols] = encoder.fit_transform(X[categorical_cols].astype(str))

# 6. Partición train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("Preprocesamiento completado.")
print(f"Dimensiones -> Train: {X_train.shape} | Test: {X_test.shape}")


In [ ]:
# Para que el benchmark sea válido en tu celda de métricas, dejamos el modelo de tu compañero corriendo aquí
print("Entrenando modelos clásicos de la Parte 1 para la comparativa...")

# Random Forest con la misma semilla para una comparación justa
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Predicciones del modelo clásico
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("Modelos de referencia listos.")

In [ ]:
# 1. Configurar token de TabPFN
os.environ["TABPFN_TOKEN"] = "tabpfn_sk_iPZbyLLnlUZKB1kKSHFg2Qu62IHXdl_NdvX8bGbL6V8"

# 2. Permitir datasets grandes
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

# 3. Importar clasificador con token activo
from tabpfn import TabPFNClassifier

print("Iniciando TabPFN...")

# 4. Inicializar y entrenar
tabpfn_model = TabPFNClassifier(ignore_pretraining_limits=True)
tabpfn_model.fit(X_train, y_train)

# 5. Predicciones
y_pred_pfn = tabpfn_model.predict(X_test)
y_prob_pfn = tabpfn_model.predict_proba(X_test)[:, 1]

print("Inferencia con TabPFN finalizada.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve

print("Calculando métricas...")

num_classes = len(np.unique(y_test))
is_multiclass = num_classes > 2

probs_rf_full = rf_model.predict_proba(X_test)
probs_pfn_full = tabpfn_model.predict_proba(X_test)

accuracy_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf, average='macro')

accuracy_pfn = accuracy_score(y_test, y_pred_pfn)
f1_pfn = f1_score(y_test, y_pred_pfn, average='macro')

if is_multiclass:
    roc_auc_rf = roc_auc_score(y_test, probs_rf_full, multi_class='ovr')
    roc_auc_pfn = roc_auc_score(y_test, probs_pfn_full, multi_class='ovr')
else:
    roc_auc_rf = roc_auc_score(y_test, probs_rf_full[:, 1])
    roc_auc_pfn = roc_auc_score(y_test, probs_pfn_full[:, 1])

print("\n" + "="*45)
print(f"{'MÉTRICA':<15} | {'RANDOM FOREST':<13} | {'TabPFN':<10}")
print("="*45)
print(f"{'Accuracy':<15} | {accuracy_rf:<13.4f} | {accuracy_pfn:<10.4f}")
print(f"{'F1-Score (Macro)':<15} | {f1_rf:<13.4f} | {f1_pfn:<10.4f}")
print(f"{'ROC-AUC':<15} | {roc_auc_rf:<13.4f} | {roc_auc_pfn:<10.4f}")
print("="*45)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_rf = confusion_matrix(y_test, y_pred_rf)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf)
disp_rf.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title("Matriz de Confusión: Random Forest")

cm_pfn = confusion_matrix(y_test, y_pred_pfn)
disp_pfn = ConfusionMatrixDisplay(confusion_matrix=cm_pfn)
disp_pfn.plot(ax=axes[1], cmap='Greens', values_format='d')
axes[1].set_title("Matriz de Confusión: TabPFN")

plt.tight_layout()
plt.show()

if is_multiclass:
    print(f"\nSe detectaron {num_classes} clases. "
          "La curva ROC tradicional solo aplica a problemas binarios.")
else:
    plt.figure(figsize=(7, 5))
    fpr_rf, tpr_rf, _ = roc_curve(y_test, probs_rf_full[:, 1])
    fpr_pfn, tpr_pfn, _ = roc_curve(y_test, probs_pfn_full[:, 1])

    plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_rf:.2f})', color='blue')
    plt.plot(fpr_pfn, tpr_pfn, label=f'TabPFN (AUC = {roc_auc_pfn:.2f})', color='green')
    plt.plot([0, 1], [0, 1], 'k--', label='Aleatorio')

    plt.xlabel('Tasa de Falsos Positivos (FPR)')
    plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
    plt.title('Comparativa de Curvas ROC')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.show()


In [ ]:
print("Preprocesando Crop Recommendation...")

if 'Unnamed: 0' in df_crop.columns:
    df_crop = df_crop.drop(columns=['Unnamed: 0'])

# Columna objetivo
target_col_crop = 'label'

print(f"Columna objetivo: '{target_col_crop}'")

df_processed_crop = df_crop.copy()
df_processed_crop = df_processed_crop.dropna(subset=[target_col_crop])

X_crop = df_processed_crop.drop(columns=[target_col_crop])
y_crop = df_processed_crop[target_col_crop]

# Codificar variable objetivo
le_crop = LabelEncoder()
y_crop = le_crop.fit_transform(y_crop)
print(f"Clases detectadas: {len(le_crop.classes_)} cultivos")

# Manejo de valores nulos
categorical_cols_crop = X_crop.select_dtypes(include=['object', 'category']).columns
numeric_cols_crop = X_crop.select_dtypes(include=['number']).columns

if len(categorical_cols_crop) > 0:
    X_crop[categorical_cols_crop] = X_crop[categorical_cols_crop].fillna('Desconocido')
if len(numeric_cols_crop) > 0:
    X_crop[numeric_cols_crop] = X_crop[numeric_cols_crop].fillna(X_crop[numeric_cols_crop].median())

# Codificar categóricas para TabPFN
if len(categorical_cols_crop) > 0:
    encoder_crop = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_crop[categorical_cols_crop] = encoder_crop.fit_transform(X_crop[categorical_cols_crop].astype(str))

# Partición train/test
X_train_crop, X_test_crop, y_train_crop, y_test_crop = train_test_split(
    X_crop, y_crop,
    test_size=0.3,
    random_state=42,
    stratify=y_crop
)

print("Preprocesamiento completado.")
print(f"Dimensiones -> Train: {X_train_crop.shape} | Test: {X_test_crop.shape}")


In [ ]:
# Para que el benchmark sea válido en tu celda de métricas, dejamos el modelo de tu compañero corriendo aquí para CROP
print("Entrenando modelos clásicos de la Parte 1 para la comparativa (Crop Recommendation)...")

# Random Forest con la misma semilla para una comparación justa en formato multiclase
rf_model_crop = RandomForestClassifier(random_state=42)
rf_model_crop.fit(X_train_crop, y_train_crop)

# Predicciones del modelo clásico para el dataset de cultivos
y_pred_rf_crop = rf_model_crop.predict(X_test_crop)

# NOTA: Al ser un problema MULTICLASE, guardamos la matriz de probabilidades completa
# (No usamos el [:, 1] del dataset binario porque aquí hay múltiples cultivos posibles)
y_prob_rf_crop = rf_model_crop.predict_proba(X_test_crop)

print("Modelos de referencia listos para Crop Dataset.")

In [ ]:
print("Iniciando TabPFN para Crop Recommendation (In-Context Learning Multiclase)...")

# 1. El token ya está configurado en el entorno de la celda anterior, no es necesario hacer borrado profundo de nuevo.
# 2. Inicializamos el clasificador para el nuevo set de datos
tabpfn_model_crop = TabPFNClassifier()

# 3. In-Context Learning: Pasamos los datos del dataset de cultivos
tabpfn_model_crop.fit(X_train_crop, y_train_crop)

# 4. Generar predicciones en el conjunto de prueba de cultivos
y_pred_pfn_crop = tabpfn_model_crop.predict(X_test_crop)

# NOTA MULTICLASE: Guardamos la matriz de probabilidades completa para todas las clases de cultivos
# (Quitamos el [:, 1] porque ya no es una clasificación binaria)
y_prob_pfn_crop = tabpfn_model_crop.predict_proba(X_test_crop)

print("Inferencia con TabPFN finalizada.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

print("Calculando métricas para Crop Dataset...")

num_classes_crop = len(np.unique(y_test_crop))
is_multiclass_crop = num_classes_crop > 2

accuracy_rf_c = accuracy_score(y_test_crop, y_pred_rf_crop)
f1_rf_c = f1_score(y_test_crop, y_pred_rf_crop, average='macro')

accuracy_pfn_c = accuracy_score(y_test_crop, y_pred_pfn_crop)
f1_pfn_c = f1_score(y_test_crop, y_pred_pfn_crop, average='macro')

if is_multiclass_crop:
    roc_auc_rf_c = roc_auc_score(y_test_crop, y_prob_rf_crop, multi_class='ovr', average='macro')
    roc_auc_pfn_c = roc_auc_score(y_test_crop, y_prob_pfn_crop, multi_class='ovr', average='macro')
else:
    roc_auc_rf_c = roc_auc_score(y_test_crop, y_prob_rf_crop[:, 1])
    roc_auc_pfn_c = roc_auc_score(y_test_crop, y_prob_pfn_crop[:, 1])

print("\n" + "="*60)
print(f"{'MÉTRICA (CROP)':<20} | {'RANDOM FOREST':<13} | {'TabPFN':<10}")
print("="*60)
print(f"{'Accuracy':<20} | {accuracy_rf_c:<13.4f} | {accuracy_pfn_c:<10.4f}")
print(f"{'F1-Score (Macro)':<20} | {f1_rf_c:<13.4f} | {f1_pfn_c:<10.4f}")
print(f"{'ROC-AUC (OVR Macro)':<20} | {roc_auc_rf_c:<13.6f} | {roc_auc_pfn_c:<10.6f}")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(22, 9))

cm_rf_c = confusion_matrix(y_test_crop, y_pred_rf_crop)
disp_rf_c = ConfusionMatrixDisplay(confusion_matrix=cm_rf_c, display_labels=le_crop.classes_)
disp_rf_c.plot(ax=axes[0], cmap='Blues', values_format='d', include_values=True)
axes[0].set_title("Matriz de Confusión Crop: Random Forest", fontsize=14, pad=15)
axes[0].set_xticklabels(le_crop.classes_, rotation=90, fontsize=10)
axes[0].set_yticklabels(le_crop.classes_, fontsize=10)

cm_pfn_c = confusion_matrix(y_test_crop, y_pred_pfn_crop)
disp_pfn_c = ConfusionMatrixDisplay(confusion_matrix=cm_pfn_c, display_labels=le_crop.classes_)
disp_pfn_c.plot(ax=axes[1], cmap='Greens', values_format='d', include_values=True)
axes[1].set_title("Matriz de Confusión Crop: TabPFN", fontsize=14, pad=15)
axes[1].set_xticklabels(le_crop.classes_, rotation=90, fontsize=10)
axes[1].set_yticklabels(le_crop.classes_, fontsize=10)

plt.tight_layout()
plt.show()

if is_multiclass_crop:
    print(f"\nSe detectaron {num_classes_crop} clases. "
          "El ROC-AUC OVR macro está calculado en la tabla.")
